# <span style="color:#1E88E5">🌐 Web Server & Application Server – Πώς «σερβίρεται» μια Python εφαρμογή</span>

---

### <span style="color:#1E88E5">❓ Τι είναι ένας web server;</span>

<div style="background:rgba(67,160,71,0.12); border-left:5px solid #43A047; padding:10px 14px; border-radius:6px;">

💡 Ένας **web server** είναι πρόγραμμα που ακούει σε μια θύρα δικτύου, δέχεται **HTTP αιτήματα** και στέλνει πίσω **HTTP απαντήσεις**.

</div>

Τι κάνει:
- Ακούει σε μια **θύρα** (port), π.χ. `80`, `443`, `8000`.
- Δέχεται συνδέσεις **TCP** (το πρωτόκολλο μεταφοράς πάνω στο οποίο «τρέχει» το HTTP).
- Κάνει **parsing** (ανάλυση) των bytes σε HTTP αίτημα: method, path, headers, body.
- Στέλνει την απάντηση σε σωστή HTTP μορφή.
- Διαχειρίζεται πολλές ταυτόχρονες συνδέσεις, **timeouts**, **keep-alive** (επαναχρησιμοποίηση σύνδεσης).

<div style="background:rgba(251,140,0,0.12); border-left:5px solid #FB8C00; padding:10px 14px; border-radius:6px;">

⚠️ Η λέξη «server» έχει **3 έννοιες**: (1) το πρόγραμμα που μιλάει HTTP, (2) όλο το backend που τρέχει (server + εφαρμογή), (3) το ίδιο το **μηχάνημα**. Από εκεί προέρχεται η σύγχυση.

</div>

---

### <span style="color:#1E88E5">❓ Το backend που γράφω είναι web server;</span>

<div style="background:rgba(67,160,71,0.12); border-left:5px solid #43A047; padding:10px 14px; border-radius:6px;">

💡 Αυστηρά μιλώντας, όχι: γράφετε την **εφαρμογή** (τη λογική), η οποία τρέχει **μέσα** σε έναν server που αναλαμβάνει όλο το δίκτυο και το HTTP.

</div>

- Η εφαρμογή σας (FastAPI, Flask, Django) ορίζει **routes** (διαδρομές, π.χ. `/users`), διαβάζει τη βάση, επιστρέφει JSON.
- **Δεν** ανοίγει sockets και **δεν** κάνει parsing HTTP. Παίρνει έτοιμο αντικείμενο αιτήματος από τον server.

Μια ASGI εφαρμογή χωρίς framework είναι απλώς μια συνάρτηση που καλεί ο server:

```python
async def app(scope, receive, send):                          # ο server την καλεί για κάθε αίτημα
    await send({"type": "http.response.start", "status": 200,  # στέλνει status code
                "headers": [(b"content-type", b"text/plain")]}) # και headers
    await send({"type": "http.response.body", "body": b"Hello"}) # στέλνει το σώμα της απάντησης
```

<div style="background:rgba(142,36,170,0.12); border-left:5px solid #8E24AA; padding:10px 14px; border-radius:6px;">

🧠 **Αναλογία:** Ο server είναι ο **σερβιτόρος** (παίρνει παραγγελίες, φέρνει πιάτα). Η εφαρμογή σας είναι η **κουζίνα** (αποφασίζει τι μαγειρεύεται). Το μηχάνημα είναι το **κτίριο**.

</div>

---

### <span style="color:#1E88E5">❓ Τι είναι το WSGI και το ASGI;</span>

<div style="background:rgba(67,160,71,0.12); border-left:5px solid #43A047; padding:10px 14px; border-radius:6px;">

💡 Είναι **πρότυπα** (standards) της Python που ορίζουν πώς ένας server **καλεί** μια εφαρμογή. Χάρη σε αυτά, οποιοσδήποτε συμβατός server τρέχει οποιαδήποτε συμβατή εφαρμογή.

</div>

| | **WSGI** | **ASGI** |
|---|---|---|
| **Πλήρες όνομα** | Web Server Gateway Interface | Asynchronous Server Gateway Interface |
| **Τρόπος λειτουργίας** | Σύγχρονος (ένα αίτημα τη φορά ανά worker) | Ασύγχρονος (`async`/`await`) |
| **WebSockets / μακριές συνδέσεις** | Όχι | Ναι |
| **Frameworks** | Flask, κλασικό Django | FastAPI, Starlette, Django (ASGI mode) |
| **Servers** | Gunicorn, uWSGI | Uvicorn, Hypercorn |

---

### <span style="color:#1E88E5">❓ Τι είναι ο Uvicorn;</span>

<div style="background:rgba(67,160,71,0.12); border-left:5px solid #43A047; padding:10px 14px; border-radius:6px;">

💡 Ο **Uvicorn** είναι **ASGI server** για Python: δέχεται HTTP αιτήματα και καλεί την ASGI εφαρμογή σας. Είναι η προεπιλογή για FastAPI.

</div>

```bash
uvicorn main:app --reload   # main = αρχείο main.py, app = το αντικείμενο της εφαρμογής, --reload = επανεκκίνηση σε κάθε αλλαγή κώδικα (μόνο για development)
```

- Γρήγορος χάρη στο **uvloop** (γρήγορο event loop για asyncio) και στο **httptools** (γρήγορο HTTP parsing).
- Σε production: πολλαπλοί **workers** (διεργασίες) με `--workers` ή μέσω Gunicorn.

<div style="background:rgba(251,140,0,0.12); border-left:5px solid #FB8C00; padding:10px 14px; border-radius:6px;">

⚠️ Ο Uvicorn **δεν** είναι framework. Δεν ορίζετε routes σε αυτόν· απλώς «σερβίρει» την εφαρμογή που γράψατε με ένα framework.

</div>

---

### <span style="color:#1E88E5">❓ Ποια η διαφορά web server και application server;</span>

<div style="background:rgba(67,160,71,0.12); border-left:5px solid #43A047; padding:10px 14px; border-radius:6px;">

💡 **Και οι δύο μιλάνε HTTP**. Η διαφορά είναι στον **ρόλο**: ο web server κάνει τις γενικές δουλειές μπροστά από όλα, ο application server **εκτελεί τον κώδικά σας**.

</div>

| | **Web server** (π.χ. Nginx) | **Application server** (π.χ. Uvicorn, Tomcat) |
|---|---|---|
| **Τρέχει κώδικα εφαρμογής;** | Όχι | Ναι, μέσα στην ίδια διεργασία |
| **Κύρια δουλειά** | TLS/HTTPS, στατικά αρχεία, load balancing, caching, προστασία | Φόρτωση και εκτέλεση της εφαρμογής |
| **Εξάρτηση από γλώσσα** | Ανεξάρτητος | Δεμένος με τη γλώσσα της εφαρμογής |
| **Επικοινωνία με τα επόμενα στρώματα** | Μέσω δικτύου (HTTP, FastCGI) | Μέσω κλήσεων συναρτήσεων (ASGI, WSGI, Servlet) |
| **Θέση στην αλυσίδα** | Μπροστά, «βλέπει» το internet | Πίσω από τον web server |

Όροι:
- **<span style="color:#43A047">Reverse proxy</span>**: server που δέχεται αιτήματα για λογαριασμό άλλων servers και τα προωθεί σε αυτούς.
- **<span style="color:#43A047">Servlet container</span>**: ο application server της Java (π.χ. Tomcat), «περιέχει» τα servlets (κλάσεις της εφαρμογής).
- **<span style="color:#43A047">Load balancing</span>**: κατανομή φορτίου σε πολλές διεργασίες/μηχανήματα.

<div style="background:rgba(142,36,170,0.12); border-left:5px solid #8E24AA; padding:10px 14px; border-radius:6px;">

🧠 **Αναλογία:** Ο web server είναι ο **υποδοχέας** στην πόρτα του εστιατορίου (καλωσορίζει, ελέγχει, μοιράζει σε τραπέζια). Ο application server είναι ο **σερβιτόρος** που μιλάει με την κουζίνα.

</div>

---

### <span style="color:#1E88E5">❓ Πώς συνδέονται μεταξύ τους σε ένα τυπικό setup;</span>

```
Browser  --HTTPS-->  Nginx  --HTTP-->  [ Uvicorn + FastAPI app ]
                   διεργασία 1          διεργασία 2 (ίδια διεργασία για server + εφαρμογή)
```

- Η εφαρμογή τρέχει **μέσα** στον application server.
- Ο web server **δεν** φιλοξενεί τον application server: είναι **ξεχωριστά προγράμματα** που μιλάνε μέσω δικτύου (HTTP ή **Unix socket**, δηλαδή τοπική επικοινωνία μέσω αρχείου στο ίδιο μηχάνημα).

<div style="background:rgba(251,140,0,0.12); border-left:5px solid #FB8C00; padding:10px 14px; border-radius:6px;">

⚠️ Ο web server μπροστά είναι **προαιρετικός**. Σε development συνήθως λείπει, και στο cloud τον ρόλο του συχνά παίζει ο load balancer της πλατφόρμας (π.χ. AWS ALB) ή το Kubernetes ingress.

</div>

**Γιατί βάζουμε Nginx μπροστά;** Ο Uvicorn είναι καλός στο να τρέχει Python, αλλά όχι στο να αντέχει απευθείας το internet (αργοί/κακόβουλοι πελάτες) ή να σερβίρει αρχεία αποδοτικά. Ο Nginx το αντίθετο.

---

### <span style="color:#1E88E5">❓ Υπάρχει διαφορετικός web / application server για κάθε γλώσσα;</span>

<div style="background:rgba(67,160,71,0.12); border-left:5px solid #43A047; padding:10px 14px; border-radius:6px;">

💡 Ό,τι επικοινωνεί μέσω **πρωτοκόλλου δικτύου** είναι **ανεξάρτητο από γλώσσα**. Ό,τι επικοινωνεί μέσω **κλήσεων συναρτήσεων** μέσα στη διεργασία είναι **δεμένο με τη γλώσσα**.

</div>

- **Web server** → ίδιος για όλες τις γλώσσες (προωθεί απλώς HTTP).
- **Application server** → ένας ανά γλώσσα, γιατί πρέπει να καλέσει τον κώδικα μέσω του προτύπου της γλώσσας:

| **Γλώσσα** | **Πρότυπο** | **Application servers** |
|---|---|---|
| **Python** | WSGI / ASGI | Gunicorn, Uvicorn |
| **Ruby** | Rack | Puma |
| **Java** | Servlet API | Tomcat, Jetty |
| **PHP** | FastCGI | PHP-FPM |
| **C# / .NET** | — | Kestrel |
| **Node.js / Go** | Ενσωματωμένος | `http` module / `net/http` |

<div style="background:rgba(251,140,0,0.12); border-left:5px solid #FB8C00; padding:10px 14px; border-radius:6px;">

⚠️ Ο application server **δεν χρειάζεται να είναι γραμμένος** στη γλώσσα της εφαρμογής. Πρέπει να είναι **συμβατός**: να φορτώνει τον runtime της (π.χ. Python interpreter) και να υλοποιεί το πρότυπο. Π.χ. ο **Granian** είναι σε Rust, ο **uWSGI** και ο **PHP-FPM** σε C.

</div>

Ένας Nginx μπορεί να εξυπηρετεί **πολλές εφαρμογές σε διαφορετικές γλώσσες** ταυτόχρονα, π.χ. `/api` → Python, `/admin` → Java.

---

### <span style="color:#1E88E5">❓ Μπορώ να χρησιμοποιήσω οποιονδήποτε web server;</span>

<div style="background:rgba(67,160,71,0.12); border-left:5px solid #43A047; padding:10px 14px; border-radius:6px;">

💡 **Ναι, σχεδόν πάντα.** Αφού μιλάνε μέσω HTTP, μπορείτε να αλλάξετε Nginx με Caddy ή Traefik **χωρίς να αγγίξετε** τον κώδικα της εφαρμογής.

</div>

Σημεία προσοχής:
- **<span style="color:#43A047">Πρωτόκολλα εκτός HTTP</span>**: ο PHP-FPM μιλάει **FastCGI** (το υποστηρίζουν Nginx, Apache, Caddy· όχι εγγενώς ο Traefik). Ο uWSGI έχει δικό του πρωτόκολλο `uwsgi`.
- **<span style="color:#43A047">WebSockets</span>**: σε κάποιους (π.χ. Nginx) χρειάζεται ρητή ρύθμιση των headers `Upgrade` και `Connection`.
- **<span style="color:#43A047">Proxy headers</span>**: πίσω από proxy η εφαρμογή «βλέπει» ως πελάτη τον proxy και νομίζει ότι η σύνδεση είναι HTTP. Ο proxy στέλνει `X-Forwarded-For` / `X-Forwarded-Proto` και ο application server πρέπει να τα εμπιστεύεται.

```bash
uvicorn main:app --proxy-headers --forwarded-allow-ips="127.0.0.1"   # εμπιστεύσου τα X-Forwarded-* headers μόνο από τον τοπικό proxy
```

<div style="background:rgba(142,36,170,0.12); border-left:5px solid #8E24AA; padding:10px 14px; border-radius:6px;">

🧠 **Συμπέρασμα:** Η επιλογή web server είναι θέμα **αναγκών και προτίμησης** (ευκολία, αυτόματο HTTPS, containers), όχι συμβατότητας με τη γλώσσα.

</div>

---

### <span style="color:#1E88E5">❓ Ποιες είναι οι πιο δημοφιλείς επιλογές;</span>

**Web servers / reverse proxies:**

- **<span style="color:#43A047">Nginx</span>**: ο πιο διαδεδομένος, γρήγορος και ελαφρύς. Η «default» επιλογή για reverse proxy.
- **<span style="color:#43A047">Apache HTTP Server</span>**: ο παλαιότερος, πολύ ευέλικτος με modules. Συχνός σε shared hosting και PHP.
- **<span style="color:#43A047">Caddy</span>**: σε Go, με **αυτόματο HTTPS** (βγάζει/ανανεώνει πιστοποιητικά μόνος του) και απλό config.
- **<span style="color:#43A047">Traefik</span>**: για containers (Docker, Kubernetes). Ανακαλύπτει αυτόματα τις υπηρεσίες.
- **<span style="color:#43A047">HAProxy</span>**: κυρίως load balancer, για πολύ μεγάλη κίνηση.

**Application servers για Python:**

- **<span style="color:#43A047">Uvicorn</span>**: ο πιο δημοφιλής ASGI server (FastAPI, Starlette).
- **<span style="color:#43A047">Gunicorn</span>**: ο κλασικός WSGI server (Flask, Django). Συχνά διαχειρίζεται πολλούς Uvicorn workers.
- **<span style="color:#43A047">Hypercorn</span>**: ASGI, με υποστήριξη HTTP/2 και HTTP/3.
- **<span style="color:#43A047">Granian</span>**: σε Rust, WSGI + ASGI, πολύ γρήγορος.
- **<span style="color:#43A047">uWSGI</span>**: ιστορικά δημοφιλής, πλέον σε **maintenance mode** (χωρίς νέα features).

<div style="background:rgba(142,36,170,0.12); border-left:5px solid #8E24AA; padding:10px 14px; border-radius:6px;">

🧠 **Ασφαλής αρχή:** **Nginx/Caddy + Uvicorn** για FastAPI · **Nginx/Caddy + Gunicorn** για Django/Flask.

</div>